# Введение в MapReduce модель на Python


In [3]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [4]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)
    
def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [5]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [6]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [7]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [8]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [9]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [10]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [11]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [12]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [13]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [14]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных. 

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [15]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*
 
mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*
 
mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL 

In [16]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str
    
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)
    
def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)
 
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication 

In [17]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])
 
def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])
      
output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(2.4586313909511714)),
 (1, np.float64(2.4586313909511714)),
 (2, np.float64(2.4586313909511714)),
 (3, np.float64(2.4586313909511714)),
 (4, np.float64(2.4586313909511714))]

## Inverted index 

In [18]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)
      
def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)
 
def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('it', ['0', '1', '2']),
 ('is', ['0', '1', '2']),
 ('what', ['0', '1']),
 ('banana', ['2']),
 ('a', ['2'])]

## WordCount

In [19]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):  
    yield (word, 1)
 
def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [20]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()
      
def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]
 
def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers
  
def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)
  
  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*
 
e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*
 
flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount 

In [21]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps
  
  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)
      
  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):  
    yield (word, 1)
 
def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)
  
# try to set COMBINER=REDUCER and look at the number of values sent over the network 
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None) 
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6)]),
 (1, [('a', 2), ('banana', 2), ('is', 18), ('it', 18), ('what', 10)])]

## TeraSort

In [22]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps
  
  def RECORDREADER(split):
    for value in split:
        yield (value, None)
      
  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])
    
def MAP(value:int, _):
  yield (value, None)
  
def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)
  
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, np.float64(0.05489434733843701)),
   (None, np.float64(0.07836689730639357)),
   (None, np.float64(0.0914240997016732)),
   (None, np.float64(0.1309187261765805)),
   (None, np.float64(0.17379584186314667)),
   (None, np.float64(0.24171172071712455)),
   (None, np.float64(0.25672585550217397)),
   (None, np.float64(0.2913379096094332)),
   (None, np.float64(0.3096236001876661)),
   (None, np.float64(0.3475021572262561)),
   (None, np.float64(0.3619138933974664)),
   (None, np.float64(0.3883314786803438)),
   (None, np.float64(0.42033058478871765)),
   (None, np.float64(0.4303295627811724)),
   (None, np.float64(0.4757182798047773)),
   (None, np.float64(0.480931331364151)),
   (None, np.float64(0.48274460771428385))]),
 (1,
  [(None, np.float64(0.6316330196323304)),
   (None, np.float64(0.6350383878334633)),
   (None, np.float64(0.6513927040494235)),
   (None, np.float64(0.6716073522581879)),
   (None, np.float64(0.7350770602337825)),
   (None, np.float64(0.7398277169025

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [37]:
from typing import Iterator

input_numbers =[15, 32, 9, 87, 42, 1, 99, 23]

def RECORDREADER():
  for i, val in enumerate(input_numbers):
    yield (i, val)

def MAP(k1: int, v1: int):
  yield ('MAX', v1)

def REDUCE(key: str, values: Iterator[int]):
  yield (key, max(values))

output = MapReduce(RECORDREADER, MAP, REDUCE)
print("Максимальное значение:", list(output))
# Должно быть: [('MAX', 99)]

Максимальное значение: [('MAX', 99)]


### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [ ]:
def RECORDREADER():
  for i, val in enumerate(input_numbers):
    yield (i, val)

def MAP(k1: int, v1: int):
  yield ('MEAN', (v1, 1))

def REDUCE(key: str, values: Iterator[tuple]):
  total_sum = 0
  total_count = 0
  for val, count in values:
    total_sum += val
    total_count += count
    
  if total_count > 0:
    yield (key, total_sum / total_count)

output = MapReduce(RECORDREADER, MAP, REDUCE)
print("Арифметическое среднее:", list(output))
# Должно быть: [('MEAN', 38.5)]

Арифметическое среднее: [('MEAN', 38.5)]


### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [38]:
def groupbykey_sort(iterable):
  sorted_iterable = sorted(iterable, key=lambda x: x[0])
  
  if not sorted_iterable:
    return []

  result =[]
  current_key = sorted_iterable[0][0]
  current_values =[]

  for k, v in sorted_iterable:
    if k == current_key:
      current_values.append(v)
    else:
      result.append((current_key, current_values))
      current_key = k
      current_values = [v]
  
  result.append((current_key, current_values))
  
  return result

test_data =[('a', 1), ('b', 2), ('a', 3), ('c', 4), ('b', 5), ('a', 6)]
print("GroupByKey (sort-based):", groupbykey_sort(test_data))
# Должно быть: [('a', [1, 3, 6]), ('b',[2, 5]), ('c', [4])]

GroupByKey (sort-based): [('a', [1, 3, 6]), ('b', [2, 5]), ('c', [4])]


### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [40]:
duplicates_list =['apple', 'banana', 'apple', 'orange', 'banana', 'kiwi', 'kiwi', 'kiwi']

def RECORDREADER():
  for i, val in enumerate(duplicates_list):
    yield (i, val)

def MAP(k1: int, v1: str):
  yield (v1, None)

def REDUCE(key: str, values: Iterator):
  yield (key, None)

output = MapReduce(RECORDREADER, MAP, REDUCE)

distinct_elements = [k for k, v in output]
print("Уникальные элементы:", distinct_elements)
# Должно быть: ['apple', 'banana', 'orange', 'kiwi']

Уникальные элементы: ['apple', 'banana', 'orange', 'kiwi']


#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [42]:
from typing import Iterator

def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

# Отношение R: id, name, age
table_R =[(1, 'Alice', 25), (2, 'Bob', 30), (3, 'Charlie', 35)]
# Отношение S: id, name, age
table_S =[(2, 'Bob', 30), (3, 'Charlie', 35), (4, 'David', 40)]

In [45]:
def RECORDREADER():
    for t in table_R: yield (None, t)

def MAP(_, t):
    if t[2] > 28:
        yield (t, t)

def REDUCE(key, values):
    yield (key, key)

print("Selection:", list(MapReduce(RECORDREADER, MAP, REDUCE)))
# Должно быть: [((2, 'Bob', 30), (2, 'Bob', 30)), ((3, 'Charlie', 35), (3, 'Charlie', 35))]

Selection: [((2, 'Bob', 30), (2, 'Bob', 30)), ((3, 'Charlie', 35), (3, 'Charlie', 35))]


### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [46]:
def RECORDREADER():
    for t in table_R: yield (None, t)

def MAP(_, t):
    t_prime = (t[1],)
    yield (t_prime, t_prime)

def REDUCE(key, values):
    yield (key, key)

print("Projection:", list(MapReduce(RECORDREADER, MAP, REDUCE)))
# Должно быть: [(('Alice',), ('Alice',)), (('Bob',), ('Bob',)), (('Charlie',), ('Charlie',))]

Projection: [(('Alice',), ('Alice',)), (('Bob',), ('Bob',)), (('Charlie',), ('Charlie',))]


### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [47]:
def RECORDREADER():
    for t in table_R + table_S: yield (None, t)

def MAP(_, t):
    yield (t, t)

def REDUCE(key, values):
    yield (key, key)

print("Union:", list(MapReduce(RECORDREADER, MAP, REDUCE)))

Union: [((1, 'Alice', 25), (1, 'Alice', 25)), ((2, 'Bob', 30), (2, 'Bob', 30)), ((3, 'Charlie', 35), (3, 'Charlie', 35)), ((4, 'David', 40), (4, 'David', 40))]


### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [48]:
def RECORDREADER():
    for t in table_R + table_S: yield (None, t)

def MAP(_, t):
    yield (t, t)

def REDUCE(key, values):
    if len(list(values)) == 2:
        yield (key, key)

print("Intersection:", list(MapReduce(RECORDREADER, MAP, REDUCE)))
# Должно быть:[((2, 'Bob', 30), (2, 'Bob', 30)), ((3, 'Charlie', 35), (3, 'Charlie', 35))]

Intersection: [((2, 'Bob', 30), (2, 'Bob', 30)), ((3, 'Charlie', 35), (3, 'Charlie', 35))]


### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [55]:
def RECORDREADER():
    for t in table_R: yield ('R', t)
    for t in table_S: yield ('S', t)

def MAP(relation, t):
    yield (t, relation)

def REDUCE(key, values):
    vals = list(values)
    if 'R' in vals and 'S' not in vals:
        yield (key, key)

print("Difference (R \\ S):", list(MapReduce(RECORDREADER, MAP, REDUCE)))
# Должно быть: [((1, 'Alice', 25), (1, 'Alice', 25))]

Difference (R \ S): [((1, 'Alice', 25), (1, 'Alice', 25))]


### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [52]:
rel_R =[('a1', 'b1'), ('a2', 'b2')]
rel_S =[('b1', 'c1'), ('b2', 'c2'), ('b1', 'c3')]

def RECORDREADER():
    for t in rel_R: yield ('R', t)
    for t in rel_S: yield ('S', t)

def MAP(relation, t):
    if relation == 'R':
        a, b = t
        yield (b, ('R', a))
    else:
        b, c = t
        yield (b, ('S', c))

def REDUCE(b, values):
    vals = list(values)
    list_R =[val for rel, val in vals if rel == 'R']
    list_S =[val for rel, val in vals if rel == 'S']
    
    for a in list_R:
        for c in list_S:
            yield (None, (a, b, c))

print("Natural Join:", list(MapReduce(RECORDREADER, MAP, REDUCE)))
# Должно быть: [(None, ('a1', 'b1', 'c1')), (None, ('a1', 'b1', 'c3')), (None, ('a2', 'b2', 'c2'))]

Natural Join: [(None, ('a1', 'b1', 'c1')), (None, ('a1', 'b1', 'c3')), (None, ('a2', 'b2', 'c2'))]


### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [53]:
rel_G =[('Group1', 10, 'x'), ('Group1', 15, 'y'), ('Group2', 5, 'z')]

def RECORDREADER():
    for t in rel_G: yield (None, t)

def MAP(_, t):
    a, b, c = t
    yield (a, b)

def REDUCE(a, values):
    yield (a, sum(values))

print("Grouping & Aggregation:", list(MapReduce(RECORDREADER, MAP, REDUCE)))
# Должно быть:[('Group1', 25), ('Group2', 5)]

Grouping & Aggregation: [('Group1', 25), ('Group2', 5)]


# 

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [ ]:
M_elements =[(0, 0, 1), (0, 1, 2), (0, 2, 3), 
              (1, 0, 4), (1, 1, 5), (1, 2, 6)]
V_elements =[(0, 10), (1, 20), (2, 30)]

def RECORDREADER_STEP1():
    for i, j, v in M_elements: yield ('M', (i, j, v))
    for j, v in V_elements: yield ('V', (j, v))

def MAP_STEP1(k1, v1):
    if k1 == 'M':
        i, j, val = v1
        yield (j, ('M', i, val))  # ключ индекс j
    elif k1 == 'V':
        j, val = v1
        yield (j, ('V', val))     # ключ индекс j

def REDUCE_STEP1(j, values):
    vals = list(values)
    
    v_val = None
    for typ, *rest in vals:
        if typ == 'V':
            v_val = rest[0]
            break
            
    if v_val is not None:
        for typ, *rest in vals:
            if typ == 'M':
                i, m_val = rest
                yield (i, m_val * v_val)

step1_output = list(MapReduce(RECORDREADER_STEP1, MAP_STEP1, REDUCE_STEP1))


def RECORDREADER_STEP2():

    for i, partial_product in step1_output:
        yield (i, partial_product)

def MAP_STEP2(i, partial_product):
    yield (i, partial_product)

def REDUCE_STEP2(i, products):
    yield (i, sum(products))

final_output = list(MapReduce(RECORDREADER_STEP2, MAP_STEP2, REDUCE_STEP2))

print("Matrix-Vector (Out-of-core) Result:", final_output)
# Должно быть: [(0, 140), (1, 320)]

Matrix-Vector (Out-of-core) Result: [(0, 140), (1, 320)]


## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$. 





In [23]:
# MapReduce model
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [26]:
import numpy as np

I = 2
J = 3
K = 4*10
small_mat = np.random.rand(I,J)
big_mat = np.random.rand(J,K)

def RECORDREADER():
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      yield ((j,k), big_mat[j,k])
      
def MAP(k1, v1):
  (j, k) = k1
  w = v1
  for i in range(small_mat.shape[0]):
    yield ((i, k), small_mat[i, j] * w)
  
def REDUCE(key, values):
  yield (key, sum(values))


Проверьте своё решение

In [27]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat) 
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

In [28]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [30]:
mat_M = np.random.rand(I,J)
mat_N = np.random.rand(J,K)

def RECORDREADER():
  for i in range(mat_M.shape[0]):
    for j in range(mat_M.shape[1]):
      yield (('M', i, j), mat_M[i,j])
      
  for j in range(mat_N.shape[0]):
    for k in range(mat_N.shape[1]):
      yield (('N', j, k), mat_N[j,k])

def MAP(k1, v1):
  matrix_name = k1[0]
  
  if matrix_name == 'M':
    _, i, j = k1
    for k in range(K):
      yield ((i, k), ('M', j, v1))
      
  elif matrix_name == 'N':
    _, j, k = k1
    for i in range(I):
      yield ((i, k), ('N', j, v1))

def REDUCE(key, values):
  # key = (i, k)
  m_elements = {}
  n_elements = {}
  
  for val in values:
    mat_name, j, val_num = val
    if mat_name == 'M':
      m_elements[j] = val_num
    elif mat_name == 'N':
      n_elements[j] = val_num
      
  total_sum = 0
  for j in m_elements.keys():
    if j in n_elements:
      total_sum += m_elements[j] * n_elements[j]
      
  yield (key, total_sum)

# Проверка
solution2 = MapReduce(RECORDREADER, MAP, REDUCE)
print("Совпадает с numpy (2 матрицы из RR)?:", np.allclose(np.matmul(mat_M, mat_N), asmatrix(solution2)))

Совпадает с numpy (2 матрицы из RR)?: True


Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER. 

In [32]:
maps = 3
reducers = 2

all_elements =[]
for i in range(I):
  for j in range(J):
    all_elements.append((('M', i, j), mat_M[i,j]))
for j in range(J):
  for k in range(K):
    all_elements.append((('N', j, k), mat_N[j,k]))

np.random.shuffle(all_elements)

def INPUTFORMAT():
  global maps
  def RECORDREADER(split):
    for record in split:
        yield record
  
  split_size = int(np.ceil(len(all_elements) / maps))
  for idx in range(0, len(all_elements), split_size):
    yield RECORDREADER(all_elements[idx:idx+split_size])

def PARTITIONER(key):
  global reducers
  return hash(key) % reducers

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None)

distributed_result_list =[]
for partition_id, partition in partitioned_output:
  distributed_result_list.extend(partition)

print("Совпадает с numpy (Distributed)?:", np.allclose(np.matmul(mat_M, mat_N), asmatrix(distributed_result_list)))

480 key-value pairs were sent over a network.
Совпадает с numpy (Distributed)?: True


Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [ ]:
import numpy as np

# размерности матриц
I = 3
J = 4
K = 3

mat_M_full = np.random.rand(I, J)
mat_N_full = np.random.rand(J, K)

elements_M =[(('M', i, j), mat_M_full[i, j]) for i in range(I) for j in range(J)]
elements_N =[(('N', j, k), mat_N_full[j, k]) for j in range(J) for k in range(K)]

# имитируем случайное подмножество: удалим около 40проц элементов случайным образом
drop_rate = 0.4
elements_M_sparse =[e for e in elements_M if np.random.rand() > drop_rate]
elements_N_sparse =[e for e in elements_N if np.random.rand() > drop_rate]

all_sparse_elements = elements_M_sparse + elements_N_sparse
np.random.shuffle(all_sparse_elements)

mat_M_sparse = np.zeros((I, J))
for (name, i, j), val in elements_M_sparse:
    mat_M_sparse[i, j] = val

mat_N_sparse = np.zeros((J, K))
for (name, j, k), val in elements_N_sparse:
    mat_N_sparse[j, k] = val

reference_solution = np.matmul(mat_M_sparse, mat_N_sparse)

maps = 4  # пусть будет 4 рекордера
reducers = 2

def INPUTFORMAT():
    global maps
    
    def RECORDREADER(split):
        for record in split:
            yield record
            
    split_size = int(np.ceil(len(all_sparse_elements) / maps))
    for idx in range(0, len(all_sparse_elements), split_size):
        yield RECORDREADER(all_sparse_elements[idx : idx + split_size])

def MAP(k1, v1):
    matrix_name = k1[0]
    if matrix_name == 'M':
        _, i, j = k1
        for k in range(K):
            yield ((i, k), ('M', j, v1))
    elif matrix_name == 'N':
        _, j, k = k1
        for i in range(I):
            yield ((i, k), ('N', j, v1))

def REDUCE(key, values):
    m_elements = {}
    n_elements = {}
    for val in values:
        mat_name, j, val_num = val
        if mat_name == 'M':
            m_elements[j] = val_num
        elif mat_name == 'N':
            n_elements[j] = val_num
            
    total_sum = 0
    for j in m_elements.keys():
        if j in n_elements:
            total_sum += m_elements[j] * n_elements[j]
    yield (key, total_sum)

def PARTITIONER(key):
    global reducers
    return hash(key) % reducers

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None)

reduce_output = flatten([partition for (partition_id, partition) in partitioned_output])

def asmatrix_sparse(output, rows, cols):
    mat = np.zeros((rows, cols))
    for ((i, k), vw) in output:
        mat[i, k] = vw
    return mat

solution_matrix = asmatrix_sparse(reduce_output, I, K)

print("\n--- Результаты ---")
print("Эталонное решение (NumPy):\n", reference_solution)
print("\nРешение MapReduce:\n", solution_matrix)
print("\nМатрицы совпадают?:", np.allclose(reference_solution, solution_matrix))

48 key-value pairs were sent over a network.

--- Результаты ---
Эталонное решение (NumPy):
 [[0.69627595 0.71956511 0.15268625]
 [0.72937157 0.         0.52000241]
 [0.43814118 0.43692573 0.00332884]]

Решение MapReduce:
 [[0.69627595 0.71956511 0.15268625]
 [0.72937157 0.         0.52000241]
 [0.43814118 0.43692573 0.00332884]]

Матрицы совпадают?: True


**Вывод**: Решение будет работать корректно 